# 第四阶段：大模型的地基 (NLP 与 Transformer 文本分类)

告别了 CNN 和图像，我们正式踏入自然语言处理（NLP）的世界。现代大语言模型（如 GPT, Qwen）全部是基于 **Transformer** 架构构建的。

本节课，我们将打通一条标准的现代 NLP 文本分类（情感分析）流水线。我们将：
1. 了解 **Tokenizer (分词器)** 是如何把人类语言变成机器能懂的数字 ID 的。
2. 一瞥 **自注意力机制 (Self-Attention)** 的核心奥秘。
3. 使用 Hugging Face 加载一个真实的、极小规模的 BERT 模型 (`bert-tiny`) 完成文本分类全链路与 ONNX 导出。

## 步骤 1-3：分词器 (Tokenizer) ➔ Dataset ➔ DataLoader

### ① 知识讲解
- **Tokenizer**：大模型并不认识汉字或英文单词。它们只认识张量。Tokenizer 就像一本巨大的密码本，把 "I love you" 切成 `["I", "love", "you"]`，然后查表变成 `[1045, 2293, 2017]`。
- **Attention Mask**：由于一个 Batch 里的句子有长有短，短句必须用 `0` 填补空位（Padding）。Attention Mask 是一串由 1 和 0 组成的遮罩，告诉模型“计算注意力时，忽略那些补 0 的假词”。

### ② 为什么这么做
传统的 NLP 需要自己手写正则分词、构建词表、处理生僻词 (OOV)。极其痛苦。现在，Hugging Face 的 `AutoTokenizer` 用 3 行代码彻底垄断并统一了这一流程。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ③ 完整代码 & ④ 代码逐行解释
model_name = "prajjwal1/bert-tiny" # 这是一个被极度压缩的微型 BERT，仅 17MB，非常适合本地学习跑通全链路
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)


# 模拟一段真实的影评数据集
raw_texts = [
    "This movie is absolutely fantastic! I loved it.",  # 正向 (1)
    "The plot was terrible and the acting was wooden.", # 负向 (0)
    "A brilliant masterpiece of modern cinema.",        # 正向 (1)
    "I wasted two hours of my life watching this garbage." # 负向 (0)
]
labels = [1, 0, 1, 0]

class NLPDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=16):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # 核心：使用 Tokenizer 批量处理文本
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,        # 太长就截断
            padding='max_length',   # 太短就补 0 补到 max_len
            max_length=self.max_len,
            return_tensors='pt'     # pt 代表 pytorch 张量
        )
        # Tokenizer 返回的是字典，张量形状是 [1, max_len]，需要 squeeze 去掉 batch 维
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(NLPDataset(raw_texts, labels, tokenizer), batch_size=2, shuffle=True)

# 取出一个 Batch 看看 Tokenizer 的魔法
sample_batch = next(iter(train_loader))
print("输入的文字 ID (input_ids):\n", sample_batch['input_ids'])
print("注意力遮罩 (attention_mask):\n", sample_batch['attention_mask'])

/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


输入的文字 ID (input_ids):
 tensor([[  101,  1037,  8235, 17743,  1997,  2715,  5988,  1012,   102,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1996,  5436,  2001,  6659,  1998,  1996,  3772,  2001,  4799,
          1012,   102,     0,     0,     0,     0]])
注意力遮罩 (attention_mask):
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])


### 💡 深度避坑与实践建议
- **⑤ 容易踩坑**：新手最容易忘记传 `attention_mask`！如果把补 0 的那些位置也扔给 Transformer 去算注意力，模型会彻底学傻（认为 0 也是一种重要的语义）。
- **⑥ 科研实验室**：NLP 数据集动辄几个 GB（比如 Wikipedia）。不能像上面这样全放到内存里，实验室会用 `datasets` 库（也是 Hugging Face 提供的）做内存映射（memory-mapping）读取。
- **⑦ 工程实践建议**：工业界处理文本分类（短文本），小巧的 BERT/DistilBERT 依然是王者，由于推理延迟极低（几毫秒），它比你强行用 GPT API 或者大模型更便宜、更快。

## 步骤 4：模型搭建 (Transformer / BERT 微调)

### ① 知识讲解与 ② 为什么这么做
在深度学习的远古时代（2017年前），大家用 RNN / LSTM 逐字阅读文本，不仅慢而且容易忘掉前面的词。
**Transformer 的核心（自注意力）**：所有词同时互相看一眼（矩阵乘法），计算谁和谁更相关。比如“The animal didn't cross the street because **it** was too tired”。模型通过自注意力计算出 `it` 应该 90% 关注 `animal`，而不是 `street`。

如今我们绝对不会自己手写 Transformer，直接拉取 `AutoModelForSequenceClassification` 即可拥有上帝视角。

In [2]:
# ③ 完整代码
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# 加载微型 BERT 模型，并直接告诉它：我这是一个 2 分类任务。
# HF 库会自动砍掉模型原始的预训练头，并随机初始化一个新的 Linear 分类头。
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
print("Tiny BERT 加载完毕！")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tiny BERT 加载完毕！


## 步骤 5-6：训练 (Train) 与评估

这里的逻辑极其优雅，因为 Hugging Face 的模型在前向传播时，**只要你把 labels 传进去，它连 Loss 都会自动帮你算好！**

In [3]:
optimizer = optim.AdamW(model.parameters(), lr=2e-5) # 注意：微调 Transformer 时，学习率通常极小 (1e-5 ~ 5e-5)

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        # 把张量移动到 GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        # 前向传播：极其优雅的 HF 接口
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        # 获取内置算好的交叉熵 Loss
        loss = outputs.loss 
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} | 平均 Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 | 平均 Loss: 0.7317
Epoch 2 | 平均 Loss: 0.6685
Epoch 3 | 平均 Loss: 0.7158
Epoch 4 | 平均 Loss: 0.7642
Epoch 5 | 平均 Loss: 0.6920


### 💡 深度避坑与实践建议
- **⑤ 容易踩坑**：微调预训练模型时，如果你用了 `lr=0.01` 这种大到夸张的学习率（以前训 CNN 常用），会直接发生**“灾难性遗忘”（Catastrophic Forgetting）**，把人家微软谷歌花几百万美金训好的知识给洗脑全毁了。必须用 `2e-5` 级别的小学习率。
- **⑥ 科研实验室**：为了极致压榨性能，不仅优化器用 AdamW，还会加上**学习率衰减策略（Linear Warmup with Cosine Decay）**，即学习率先缓慢上升，再余弦下降。

## 步骤 7-10：保存、加载、推理与结果

Hugging Face 提供了一套专用的 `save_pretrained` 和 `from_pretrained`，它不仅存权重，还把模型配置 (config.json) 一起存了，这是大模型时代的黄金准则。

In [4]:
# 保存模型与 Tokenizer
save_dir = "./saved_tiny_bert"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"大模型权重与词表已完整保存在 {save_dir} 中")

# 重新加载
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_dir).to(device)
loaded_model.eval()

# --- 真实的推理测试 ---
test_text = "This is an amazing and incredibly touching film."
inputs = tokenizer(test_text, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    # 推理时没有 labels 参数，输出只包含 logits
    outputs = loaded_model(**inputs)
    # Logits 形状为 [1, 2]，取最大值的索引即可
    prediction = outputs.logits.argmax(dim=-1).item()

class_names = ["负面 (Negative)", "正面 (Positive)"]
print(f"\n📝 测试输入: '{test_text}'")
print(f"🤖 模型预测结果: {class_names[prediction]}")

大模型权重与词表已完整保存在 ./saved_tiny_bert 中

📝 测试输入: 'This is an amazing and incredibly touching film.'
🤖 模型预测结果: 正面 (Positive)


## 步骤 11：ONNX 导出与部署

NLP 模型的 ONNX 导出有一个难点：输入必须是两个张量 `input_ids` 和 `attention_mask`，并且我们需要让**句子长度维度变成动态的**，这样部署时模型才能接受任意长度的话语。

In [5]:
# 造一组假数据 (Batch=1, SeqLen=16)
dummy_input_ids = torch.zeros(1, 16, dtype=torch.long, device=device)
dummy_attention_mask = torch.ones(1, 16, dtype=torch.long, device=device)

onnx_path = "tiny_bert.onnx"

torch.onnx.export(
    loaded_model, 
    (dummy_input_ids, dummy_attention_mask), # NLP 模型有多个输入，必须打包成 Tuple
    onnx_path,
    export_params=True,
    opset_version=14,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    # 重点：同时将 batch_size 维度 (0) 和 seq_len 维度 (1) 设为动态
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'seq_len'},
        'attention_mask': {0: 'batch_size', 1: 'seq_len'},
        'logits': {0: 'batch_size'}
    }
)

print(f"NLP 模型成功导出为动态长度的 ONNX: {onnx_path}")

/var/folders/4x/gykrz2md4nz_j7l5r35_m6880000gn/T/ipykernel_33855/125297998.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0717 14:21:50.952000 33855 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `BertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/onnxscript/ve

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/Users/zoupray/miniforge3/envs/llm_env/lib/python3.10/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: seq_len will not be used, since it shares the same shape constraints with another axis: seq_len.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


[torch.onnx] Optimize the ONNX graph... ✅
NLP 模型成功导出为动态长度的 ONNX: tiny_bert.onnx


## 🏆 第四阶段结语
祝贺你正式踏入了大模型的领地！在这里你见识到了现代大模型架构的两把神器：
1. **Hugging Face (`AutoTokenizer` / `AutoModel`)**：一统江湖的开源库。
2. **Transformer 的降维打击**：不需要写复杂的网络，下载、喂数据、它自动帮你算 Loss。

请立刻在你的 `llm_env` 中跑通这套代码。如果有遇到任何关于 Hugging Face 连接报错或者网络问题，请及时告诉我。下一关，我们将把刚刚学到的【图像】与【文本】做终极融合：**第五阶段 CLIP（以文搜图的多模态对齐）**！